In [1]:
import numpy as np                                                                        
import matplotlib.pyplot as plt
import pyCloudy as pc
import pyneb as pn
from astropy    import constants as const
from astropy.io import ascii
import pandas as pd
from scipy import interpolate
import warnings
from scipy.integrate import quad, IntegrationWarning
import scipy.integrate as integrate
from numpy import log10, exp
import os

kpc = const.kpc.cgs.value

atom = ['Lya','HeII','CIV']

def read_file(path_way, atom):
    Mod = pc.CloudyModel(path_way)
    Mod.ionic_names

    N_H = sum(Mod.dr*Mod.nH)
    frac_He =1.00E-01
    frac_C = 2.45E-04
    frac_O = 4.90E-04
    frac_N = 8.51E-05
    frac_Mg = 3.47E-05

    N_HI = sum(Mod.dr*Mod.nH*Mod.get_ionic('H',0))
    N_HII = sum(Mod.dr*Mod.nH*Mod.get_ionic('H',1))
    N_HeII = frac_He*sum(Mod.dr*Mod.nH*Mod.get_ionic('He',1))
    N_OVI = frac_O*sum(Mod.dr*Mod.nH*Mod.get_ionic('O',5))
    N_NV = frac_N*sum(Mod.dr*Mod.nH*Mod.get_ionic('N',4))
    N_CIV = frac_C*sum(Mod.dr*Mod.nH*Mod.get_ionic('C',3))

    num = len(Mod.nH)
    r_CIV = path_way +  '.ele_C'
    f = open(r_CIV,'r')
    header = f.readline()
    CIV_frac = np.zeros(num)
    i = 0
    for line in f:
        line = line.strip()
        columns = line.split()
        j = float(columns[4])
        CIV_frac[i] = j
        i = i + 1

    r_He = path_way +  '.ele_He'
    f = open(r_He,'r')
    header = f.readline()
    HeII_frac = np.zeros(num)
    i = 0
    for line in f:
        line = line.strip()
        columns = line.split()
        j = float(columns[2])
        HeII_frac[i] = j
        i = i + 1



    n_H= Mod.nH
    n_He = n_H*frac_He
    n_C = n_H*frac_C
    nden_CIV = CIV_frac*n_C
    nden_HeII = HeII_frac*n_He



    if atom == 'CIV':
        Cloudy_Lum = float(Mod.get_emis_vol('C__4_154819A')) + float(Mod.get_emis_vol('C__4_155078A'))
        Cloudy_emis = (Mod.get_emis('C__4_154819A')) + (Mod.get_emis('C__4_155078A'))
        Cloudy_den = nden_CIV
    elif atom == 'Lya':
        Cloudy_Lum= float(Mod.get_emis_vol('H__1_121567A'))
        Cloudy_emis = Mod.get_emis('H__1_121567A')
        Cloudy_den = n_H
    elif atom == 'HeII':
        Cloudy_Lum = float(Mod.get_emis_vol('HE_2_164043A'))
        Cloudy_emis = Mod.get_emis('HE_2_164043A')
        Cloudy_den = nden_HeII
    return Cloudy_Lum , Cloudy_emis ,Cloudy_den

def radius(path, atom):
    Mod = pc.CloudyModel(path)
    radius = Mod.radius/kpc
    radius_kpc =Mod.radius 
    dr = Mod.dr 
    return radius, radius_kpc, dr

def make_data_file(path,atom,Lumin,idx,metals,Column_density_order):
    if idx == 1 :
        mode = "W"
    elif idx == 2 :
        mode = "WO"

    lum ,emis ,den = read_file(path,atom)
    radius_R , radius_kpc , dr=  radius(path,atom)
    tt =  pd.DataFrame(np.column_stack((radius_R,emis,den)))
    

    lum = int(Lumin*10)
    col = int(Column_density_order*10)

    if metals < 1:
        metals_int = int(metals * 1000)   # 0.001 → 1, 0.01 → 10 등
        metals_str = f"{metals_int:04d}"  # 1 → '0001', 10 → '0010'
    else:
        metals_str = str(int(metals))     # 1.0 → '1', 2.0 → '2'

    folder_name = f'{atom}L{lum}M{metals_str}NH{col}'
    folder_path = f'/home/jin/RT/RT_Run_data/{folder_name}'
    
    try:
        os.makedirs(folder_path, exist_ok=True)
        print(f"Created folder: {folder_path}")
    except Exception as e:
        print(f"Error creating folder {folder_path}: {e}")


    # 파일 저장
    # tt.to_csv(f'/home/jin/RT/CLOUDY_new_Data/{mode}_LT_{atom}_cloudy_Lumin_{Lumin}_metal_{metals}_N_H_{Column_density_order}.txt', sep='\t',index=False,header =False)
    tt.to_csv(f'/home/jin/RT/RT_Run_data/{folder_name}.txt', sep='\t',index=False,header =False)
    file2 = f'/home/jin/RT/RT_Run_data/{folder_name}/radius_emis_number_density.txt'
    header_text = f"# ATOM = {atom}, log(nuLnu) = {Lumin}  [erg/s] at 1 Ryd, log(N_H) = {Column_density_order} [1/cm2], metallicity = {metals} Z_sun, {mode} line transfer\n # (1) radius [kpc] (2) emissivity [erg/s/cm3/sr] (3) number_density [1/cm3]\n"
    with open(file2, "w") as f:
        f.write(header_text)
    tt.to_csv(file2, sep="\t", index=False, header=False, mode="a")
    # tt.to_csv(f'/home/jin/RT/RT_scat/{folder_name}.txt', sep='\t',index = False,header=False)
    

    

    
    return print("make data file!")

# CLOUDY_data(Lumin,idx_2,metals,Column_density_order,mode)


warnings.filterwarnings("ignore", category=IntegrationWarning)



In [3]:
from CLOUDY_Function import *
# Lumin = 42.0 # 37.0 - 44.0 + 41.5, 42.5, 43.5 and 44.5
# idx_1 = 1 # 1 "" and  2 "no line transfer"
idx_2 = 2
# metals = 1.0 # 0.001 - 10.0
Column_density_order = 20.5  # 22.0,22.5  # 18.0 - 25.0
mode = 'WO' # 'WO' - 'W'

Metal = np.array([0.1,1.0])
Luminosity = np.array([41.0,41.5,43.0,43.5,44.0,44.5,46.0,46.5])
for jj, Lumin in enumerate(Luminosity):
    for ii, mm in enumerate(Metal):

    # path_way_1 = path(Lumin,idx_1,metals,Column_density_order)
    # path_cloudy_1 = os.path.join(path_way_1, f'CIV_QSO')


        path_way_2 = path(Lumin,idx_2,mm,Column_density_order)
        path_cloudy_2 = os.path.join(path_way_2, f'CIV_QSO')

        atom = 'CIV'
        # make_data_file(path_cloudy_1,atom ,Lumin,idx_1,metals,Column_density_order)
        make_data_file(path_cloudy_2,atom ,Lumin,idx_2,mm,Column_density_order)



        atom = 'HeII'
        # make_data_file(path_cloudy_1,atom ,Lumin,idx_1,metals,Column_density_order)
        make_data_file(path_cloudy_2,atom ,Lumin,idx_2,mm,Column_density_order)


# make_data_file(path_LT,'HeII')

Created folder: /home/jin/RT/RT_Run_data/CIVL410M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL410M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL410M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL410M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL415M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL415M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL415M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL415M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL430M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL430M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL430M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/HeIIL430M1NH205
make data file!
Created folder: /home/jin/RT/RT_Run_data/CIVL435M0100NH205
make data file!
Created folder: /home/jin/RT/RT_Run_d

Send_files